In [1]:
!pip install safetensors transformers rich tyro
!pip uninstall protobuf
!pip install protobuf==3.20.3


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 13.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.0/462.0 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 166.1 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.0/468.0 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.7/781.7 kB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 26.9 MB/s eta 0:00:00
  Attempting uninstall: typin

In [2]:
# Code to install Unsloth, Triton, Torch etc
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton
!pip install --no-deps cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 174.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 MB 97.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.1/342.1 kB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 44.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.1/107.1 kB 917.5 kB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: python -m pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 1.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 3.6 MB/s eta 0:00:

In [3]:
# Helpful functions used through the entire notebook
import torch
import torch.nn as nn
from transformers import set_seed
import time
import inspect
import os
major_version, minor_version = torch.cuda.get_device_capability()
HAS_BFLOAT16 = (major_version >= 8)
from inspect import currentframe as _C, getframeinfo
_F = lambda c: getframeinfo(c).lineno # Gets line number
WARN = lambda x: print(f"\033[31m{x}\033[0m") # Red colored warnings

# https://stackoverflow.com/questions/18425225/getting-the-name-of-a-variable-as-a-string
def NAME(var):
    callers_local_vars = inspect.currentframe().f_back.f_locals.items()
    names = [var_name for var_name, var_val in callers_local_vars if var_val is var]
    return names[0] if len(names) != 0 else ""

def assert_same(x, y, line, dtype):
    assert(x.dtype == dtype)
    try: torch.testing.assert_close(x, y, check_stride = True)
    except Exception as error:
        raise RuntimeError(
            f"Failed allclose at line [{line}]: {NAME(x)}, {NAME(y)}\n{str(error)}"
        )

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [9]:
import torch
from torch.nn import CrossEntropyLoss
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------------------------------------------------
# MemoryEfficientLinear:
# A custom autograd Function that avoids storing large intermediate logits.
class MemoryEfficientLinear(torch.autograd.Function):
    @staticmethod
    def forward(ctx, X, weight, bias, labels, forward_function, chunk_size=8192):
        ctx.save_for_backward(X, weight, bias, labels)
        ctx.forward_function = forward_function
        ctx.chunk_size = chunk_size
        ctx.V = weight.size(0)  # Store total vocab size
        
        total_loss = 0
        V = ctx.V
        
        for start in range(0, V, chunk_size):
            end = min(start + chunk_size, V)  # Handle last chunk
            chunk_weight = weight[start:end]
            chunk_bias = bias[start:end] if bias is not None else None
            
            logits = F.linear(X, chunk_weight, chunk_bias)
            chunk_loss = forward_function(logits, labels, start, end)
            
            # Weight by actual chunk size
            total_loss += chunk_loss * (end - start)
            
        return total_loss / V  # Normalize by total vocab

    @staticmethod
    def backward(ctx, grad_output):
        X, weight, bias, labels = ctx.saved_tensors
        chunk_size = ctx.chunk_size
        V = ctx.V
        
        grad_X = torch.zeros_like(X) if X.requires_grad else None
        grad_weight = torch.zeros_like(weight)
        grad_bias = torch.zeros_like(bias) if bias is not None else None
        
        for start in range(0, V, chunk_size):
            end = min(start + chunk_size, V)  # Handle last chunk
            chunk_weight = weight[start:end]
            chunk_bias = bias[start:end] if bias is not None else None
            
            with torch.enable_grad():
                logits = F.linear(X, chunk_weight, chunk_bias)
                loss = ctx.forward_function(logits, labels, start, end)
                
                # Scale loss by chunk proportion
                scaled_loss = loss * (end - start) / V
                
            grads = torch.autograd.grad(
                scaled_loss,
                [chunk_weight, chunk_bias, X] if X.requires_grad else [chunk_weight, chunk_bias],
                grad_outputs=grad_output,
                allow_unused=True
            )
            
            # Accumulate gradients
            if grads[0] is not None:
                grad_weight[start:end] += grads[0]
            if grad_bias is not None and grads[1] is not None:
                grad_bias[start:end] += grads[1]
            if X.requires_grad and len(grads) > 2:
                grad_X += grads[2]
                
        return grad_X, grad_weight, grad_bias, None, None, None

def cross_entropy_forward(logits, labels, start, end):
    # Create mask within the function
    mask = (labels >= start) & (labels < end)
    targets = torch.where(mask, labels - start, -100)
    return F.cross_entropy(logits, targets, ignore_index=-100, reduction='mean')

def mse_forward(logits, labels, start, end):
    # Create mask within the function
    mask = (labels >= start) & (labels < end)
    targets = torch.zeros_like(logits)
    targets[mask] = 1.0  # Example target
    return F.mse_loss(logits, targets)

def transformation_function(X, linear, labels, forward_func=cross_entropy_forward):
    return MemoryEfficientLinear.apply(
        X.view(-1, X.size(-1)),
        linear.weight,
        linear.bias if linear.bias is not None else None,  # Explicit None check
        labels.view(-1),
        forward_func
    )

def validate_efficiency():
    import gc
    from torch.nn import CrossEntropyLoss
    
    # Test configuration
    bsz, qlen, hd, vocab = 4, 4096, 4096, 128000
    X = torch.randn(bsz, qlen, hd, dtype=torch.bfloat16).cuda().requires_grad_(True)
    linear = nn.Linear(hd, vocab).cuda().bfloat16()
    labels = torch.randint(0, vocab, (bsz, qlen)).cuda()

    # Baseline measurement - FIXED DIMENSIONS
    torch.cuda.reset_peak_memory_stats()
    logits_normal = linear(X).float().view(-1, vocab)  # Reshape to [bsz*qlen, vocab]
    loss_normal = F.cross_entropy(logits_normal, labels.view(-1))  # Targets [bsz*qlen]
    loss_normal.backward()
    mem_normal = torch.cuda.max_memory_allocated()
    grad_normal = linear.weight.grad.clone()
    linear.zero_grad()
    
    # Our implementation - FIXED DIMENSIONS
    torch.cuda.reset_peak_memory_stats()
    loss_efficient = transformation_function(
        X.view(bsz, qlen, hd),  # Ensure original dimensions
        linear,
        labels.view(bsz, qlen)  # Ensure original dimensions
    )
    loss_efficient.backward()
    mem_efficient = torch.cuda.max_memory_allocated()
    grad_efficient = linear.weight.grad.clone()
    linear.zero_grad()

    # Criterion 1: VRAM reduction
    vram_reduction = (mem_normal - mem_efficient) / mem_normal
    print(f"VRAM reduction: {vram_reduction:.1%} (Target >50%)")
    
    # Criterion 2: No float32 upcast
    print(f"Using dtype: {loss_efficient.dtype} (Should be bfloat16)")
    
    # Criterion 3: CE Loss works
    loss_diff = torch.abs(loss_normal - loss_efficient).item()
    print(f"Loss difference: {loss_diff:.2e} (Should be <1e-3)")
    
    # Criterion 4: Other functions work
    try:
        loss_mse = transformation_function(X, linear, labels, mse_forward)
        loss_mse.backward()
        print("Other functions: ✓")
    except Exception as e:
        print(f"Other functions: ✗ ({str(e)})")

    # Criterion 5: Dynamic chunk sizes
    try:
        for chunk_size in [4096, 8192, 16384]:
            MemoryEfficientLinear.apply(X, linear.weight, None, labels, cross_entropy_forward, chunk_size).backward()
        print("Dynamic chunks: ✓")
    except:
        print("Dynamic chunks: ✗")

    # Criterion 6: Gradient matching
    grad_error = torch.max(torch.abs(grad_normal - grad_efficient)).item()
    print(f"Gradient error: {grad_error:.2e} (Should be <1e-3)")

    # Cleanup
    del X, linear, labels
    gc.collect()
    torch.cuda.empty_cache()

# Run validation
validate_efficiency()


VRAM reduction: 46.2% (Target >50%)
Using dtype: torch.bfloat16 (Should be bfloat16)
Loss difference: 2.87e+00 (Should be <1e-3)
Other functions: ✓
Dynamic chunks: ✗
Gradient error: 4.60e-04 (Should be <1e-3)
